# SpaceX Falcon 9 – EDA with SQL
Load the launch data into SQLite and answer ten questions with SQL.

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect("my_data1.db")
df = pd.read_csv("data/dataset_part_2.csv")
df.to_sql("SPACEXTBL", con, if_exists="replace", index=False)
def q(sql):
    return pd.read_sql(sql, con)

## Task 1: Names of the unique launch sites

In [2]:
q("""SELECT DISTINCT LaunchSite FROM SPACEXTBL""")

,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


## Task 2: Five records where launch sites begin with 'CCA'

In [3]:
q("""SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5""")

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0


## Task 3: Total payload mass carried by boosters from each launch site

In [4]:
q("""SELECT LaunchSite, ROUND(SUM(PayloadMass)) AS Total_Payload_kg FROM SPACEXTBL GROUP BY LaunchSite""")

,LaunchSite,Total_Payload_kg
0,CCAFS SLC 40,305151.0
1,KSC LC 39A,167342.0
2,VAFB SLC 4E,76953.0


## Task 4: Average payload mass carried by Block 1 boosters

In [5]:
q("""SELECT ROUND(AVG(PayloadMass), 1) AS Avg_Payload_kg FROM SPACEXTBL WHERE Block = 1""")

,Avg_Payload_kg
0,2578.8


## Task 5: Date of the first successful landing on a ground pad

In [6]:
q("""SELECT MIN(Date) AS First_Ground_Pad_Landing FROM SPACEXTBL WHERE Outcome = 'True RTLS'""")

,First_Ground_Pad_Landing
0,2015-12-22


## Task 6: Boosters with drone-ship success and payload between 4,000 and 6,000 kg

In [7]:
q("""SELECT DISTINCT Serial FROM SPACEXTBL WHERE Outcome = 'True ASDS' AND PayloadMass > 4000 AND PayloadMass < 6000""")

,Serial
0,B1022
1,B1026
2,B1021
3,B1031
4,B1046
5,B1059


## Task 7: Total number of successful and failed landings

In [8]:
q("""SELECT CASE WHEN Class = 1 THEN 'Success' ELSE 'Failure' END AS Result, COUNT(*) AS Total FROM SPACEXTBL GROUP BY Class""")

,Result,Total
0,Failure,30
1,Success,60


## Task 8: Boosters that carried the maximum payload mass (subquery)

In [9]:
q("""SELECT DISTINCT Serial, PayloadMass FROM SPACEXTBL WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL)""")

,Serial,PayloadMass
0,B1048,15600.0
1,B1051,15600.0


## Task 9: Failed drone-ship landings in 2015, by month

In [10]:
q("""SELECT substr(Date, 6, 2) AS Month, Outcome, Serial, LaunchSite FROM SPACEXTBL WHERE Outcome = 'False ASDS' AND substr(Date, 1, 4) = '2015'""")

,Month,Outcome,Serial,LaunchSite
0,01,False ASDS,B1012,CCAFS SLC 40
1,04,False ASDS,B1015,CCAFS SLC 40


## Task 10: Rank landing outcomes between 2010-06-04 and 2017-03-20

In [11]:
q("""SELECT Outcome, COUNT(*) AS Total FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Outcome ORDER BY Total DESC""")

,Outcome,Total
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2
